<a href="https://colab.research.google.com/github/idoschw3/Corpus2GeoRAG/blob/main/ApplyNER/NER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [44]:
from transformers import pipeline
from tqdm.auto import tqdm
from pathlib import Path
import pandas as pd
import numpy as np
import subprocess
import tempfile
import shutil
import torch
import math
import json
import time
import csv
import os
import io


if os.name == "nt":
    userdata = None
else:
    from google.colab import userdata

### Clone Corpus2GeoRAG Repository

In [23]:
IS_LOCAL = os.name == "nt"

LOCAL_REPO_PATH = Path.home() / "Corpus2GeoRAG"
repo_dir = (
    LOCAL_REPO_PATH
    if IS_LOCAL
    else Path("/content/Corpus2GeoRAG")
)

REPO_PATH = str(repo_dir)

if not repo_dir.exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/EtzionR/Corpus2GeoRAG.git",
            REPO_PATH
        ],
        check=True
    )
elif not (repo_dir / ".git").exists():
    raise RuntimeError(f"{REPO_PATH} exists but is not a Git checkout.")

os.chdir(REPO_PATH)

print("Runtime:", "Local Windows" if IS_LOCAL else "Colab VM")
print("Working folder:", REPO_PATH)

Runtime: Local Windows
Working folder: C:\Users\idosc\Corpus2GeoRAG


### Data Loading and Initial Inspection

**Purpose:** To load the raw text data, which is crucial for subsequent NER tasks, from a JSON file. This section also includes an example to visually inspect the structure and content of a single entry, ensuring the data is correctly loaded and accessible.


**Input/Output:** Input is `/content/Corpus2GeoRAG/examples/DATA.json`.

**Date:** 2026-09-23

In [24]:
PATH = Path(REPO_PATH) / "examples" / "DATA.json"
PATH

WindowsPath('C:/Users/idosc/Corpus2GeoRAG/examples/DATA.json')

In [25]:
with open(PATH, "r", encoding="utf-8") as file:
    data = json.load(file)

len(data)

897

In [26]:
[*data.keys()][:10]

['12 (song)',
 '14 May 2026 Russian strikes on Ukraine',
 '15th BRICS summit',
 '16 March 2022 Chernihiv breadline attack',
 '17 November 2024 Russian strikes on Ukraine',
 '18 March 2022 Mykolaiv military quarters attack',
 '1st European Political Community Summit',
 '2000 Meters to Andriivka',
 '2014 pro-Russian unrest in Ukraine',
 '2020s European re-armament']

In [27]:
example = np.random.choice([*data],1)[0]


print(f'PAGE: {example}:\n\n{data[example]}')


PAGE: Torture of Russian soldiers in Mala Rohan:

The torture of Russian soldiers in Mala Rohan was an incident during the 2022 Russian invasion of Ukraine that occurred in the village of Mala Rohan. As documented by the UN High Commissioner for Human Rights, members of the Ukrainian Armed Forces shot three captured Russian soldiers in the legs and tortured Russian soldiers who were wounded. The incident is reported to have occurred on the evening of March 25, 2022 and was first reported following the publication on social media of a video of unknown authorship between March 27 and March 28. As a case of summary execution and torture of prisoners of war, the incident might qualify as a war crime.


== Video ==
On the morning of March 27, two videos were posted on Reddit and Twitter, and a more complete version, two minutes longer, later appeared on YouTube. The Times reported that the videos had been promoted by Maria Dubovikova of the Russian International Affairs Council, but their o

In [28]:
TITLE = '=='
MIN_LENGTH = 5

paragraphs = data[example].split('\n')
paragraphs_processed = [text for text in paragraphs if text.startswith(TITLE)==False and len(text)>=MIN_LENGTH]

print(f'{len(paragraphs_processed)} Paragraphs Extracted from {len(paragraphs)}\n{round(len(paragraphs_processed)/len(paragraphs)*100,1)}%')

17 Paragraphs Extracted from 32
53.1%


### Text Preprocessing: Paragraph Filtering

**Purpose:** To perform an initial cleaning of the raw text by filtering out structural elements and short, uninformative lines. This step aims to prepare cleaner text segments for downstream NER processing by removing elements such as section titles and empty or minimal lines.

**Date:** 2026-09-24

In [29]:
TITLE = '=='
MIN_LENGTH = 5

data_processed = {}

total_paragraphs = 0
total_processed = 0

for key, text in data.items():
  paragraphs = text.split('\n')
  paragraphs_processed = [text for text in paragraphs if text.startswith(TITLE)==False and len(text)>=MIN_LENGTH]
  data_processed[key] = paragraphs_processed

  total_paragraphs += len(paragraphs)
  total_processed += len(paragraphs_processed)

print(f'{total_processed} paragraphs extracted from {total_paragraphs}\n {round(total_processed/total_paragraphs*100,1)}%')

42822 paragraphs extracted from 88029
 48.6%


In [30]:
example = np.random.choice([*data_processed],1)[0]

print(f'PAGE: {example}:\n\n{data_processed[example]}')

PAGE: Zapad 2021:

['Zapad 2021 (Russian: Запад-2021, English: West 2021) was a joint strategic exercise between the armed forces of the Russian Federation and Belarus, which took place from 10 to 15 September 2021. According to the Ministry of Defence of the Russian Federation, approximately 200,000 military personnel, up to 760 pieces of equipment, and 15 ships participated in the exercises.', 'According to some international experts, the exercise was part of Russian preparation for the subsequent invasion of Ukraine.', 'Strategic exercises between Belarus and Russia are held every two years, as per an agreement reached between Russian President Dmitry Medvedev and Belarusian President Alexander Lukashenko in 2009. In 2021, they were combined with maneuvers of the Collective Rapid Reaction Forces of the CSTO member states.', 'According to First Deputy Minister of Defense of the Republic of Belarus, Major General Viktor Gulevich, "Zapad-2021 is purely defensive, and its holding does n

In [31]:
SAVE_PATH = Path(REPO_PATH) / "ApplyNER" / "data_initial_cleaning.json"
SAVE_PATH.parent.mkdir(parents=True, exist_ok=True)

with open(SAVE_PATH, "w", encoding="utf-8") as f:
    json.dump(data_processed, f, ensure_ascii=False, indent=2)

print(f"Saved to: {SAVE_PATH}")

Saved to: C:\Users\idosc\Corpus2GeoRAG\ApplyNER\data_initial_cleaning.json


## Initial NER application

**Purpose:** To perform an initial NER application with base models and analyze the results

**Date:** 2026-09-24

### example

#### DistilBERT

In [32]:
MODEL_DistilBERT = "dslim/distilbert-NER"

ner_DistilBERT = pipeline(
    "ner",
    model=MODEL_DistilBERT,
    aggregation_strategy="average",
    device=0 if torch.cuda.is_available() else -1
)

print("Model device:", ner_DistilBERT.device)

ner_DistilBERT

Loading weights: 100%|██████████| 102/102 [00:00<00:00, 473.83it/s]


Model device: cuda:0


TokenClassificationPipeline: {'model': 'DistilBertForTokenClassification', 'dtype': 'float32', 'device': 'cuda', 'input_modalities': 'text'}

In [33]:
output_DistilBERT = ner_DistilBERT(data_processed[example])
output_DistilBERT

[[{'entity_group': 'MISC',
   'score': np.float32(0.53940827),
   'word': 'Zapad',
   'start': 0,
   'end': 5},
  {'entity_group': 'MISC',
   'score': np.float32(0.839036),
   'word': 'Russian',
   'start': 12,
   'end': 19},
  {'entity_group': 'MISC',
   'score': np.float32(0.7054061),
   'word': 'English',
   'start': 33,
   'end': 40},
  {'entity_group': 'LOC',
   'score': np.float32(0.74097437),
   'word': 'West',
   'start': 42,
   'end': 46},
  {'entity_group': 'LOC',
   'score': np.float32(0.9661194),
   'word': 'Russian Federation',
   'start': 116,
   'end': 134},
  {'entity_group': 'LOC',
   'score': np.float32(0.99835545),
   'word': 'Belarus',
   'start': 139,
   'end': 146},
  {'entity_group': 'ORG',
   'score': np.float32(0.97504026),
   'word': 'Ministry of Defence',
   'start': 212,
   'end': 231},
  {'entity_group': 'LOC',
   'score': np.float32(0.9573382),
   'word': 'Russian Federation',
   'start': 239,
   'end': 257}],
 [{'entity_group': 'MISC',
   'score': np.floa

In [34]:
df_DistilBERT = pd.DataFrame([
    {
        "page": example,
        "paragraph_id": paragraph_id,
        "paragraph": data_processed[example][paragraph_id],
        **entity
    }
    for paragraph_id, entities in enumerate(output_DistilBERT)
    for entity in entities
])

df_DistilBERT

,page,paragraph_id,paragraph,entity_group,score,word,start,end
0,Zapad 2021,0,"Zapad 2021 (Russian: Запад-2021, English: West...",MISC,0.539408,Zapad,0,5
1,Zapad 2021,0,"Zapad 2021 (Russian: Запад-2021, English: West...",MISC,0.839036,Russian,12,19
2,Zapad 2021,0,"Zapad 2021 (Russian: Запад-2021, English: West...",MISC,0.705406,English,33,40
3,Zapad 2021,0,"Zapad 2021 (Russian: Запад-2021, English: West...",LOC,0.740974,West,42,46
4,Zapad 2021,0,"Zapad 2021 (Russian: Запад-2021, English: West...",LOC,0.966119,Russian Federation,116,134
...,...,...,...,...,...,...,...,...
170,Zapad 2021,20,Warsaw-based Belarusian former politician and ...,MISC,0.943980,based,7,12
171,Zapad 2021,20,Warsaw-based Belarusian former politician and ...,MISC,0.993857,Belarusian,13,23
172,Zapad 2021,20,Warsaw-based Belarusian former politician and ...,PER,0.999114,Andrej Sannikau,56,71
173,Zapad 2021,20,Warsaw-based Belarusian former politician and ...,LOC,0.503009,Zapad,89,94


#### BERT

In [35]:
MODEL_BERT = "dslim/bert-base-NER"

ner_BERT = pipeline(
    "ner",
    model=MODEL_BERT,
    aggregation_strategy="average",
    device=0 if torch.cuda.is_available() else -1
)

print("Model device:", ner_BERT.device)

ner_BERT

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 560.97it/s]
[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model device: cuda:0


TokenClassificationPipeline: {'model': 'BertForTokenClassification', 'dtype': 'float32', 'device': 'cuda', 'input_modalities': 'text'}

In [36]:
output_BERT = ner_BERT(data_processed[example])
output_BERT

[[{'entity_group': 'MISC',
   'score': np.float32(0.39069656),
   'word': 'Zapad',
   'start': 0,
   'end': 5},
  {'entity_group': 'MISC',
   'score': np.float32(0.99676335),
   'word': 'Russian',
   'start': 12,
   'end': 19},
  {'entity_group': 'LOC',
   'score': np.float32(0.457594),
   'word': 'West',
   'start': 42,
   'end': 46},
  {'entity_group': 'LOC',
   'score': np.float32(0.99949753),
   'word': 'Russian Federation',
   'start': 116,
   'end': 134},
  {'entity_group': 'LOC',
   'score': np.float32(0.99979705),
   'word': 'Belarus',
   'start': 139,
   'end': 146},
  {'entity_group': 'ORG',
   'score': np.float32(0.9896689),
   'word': 'Ministry of Defence of the Russian',
   'start': 212,
   'end': 246},
  {'entity_group': 'LOC',
   'score': np.float32(0.961596),
   'word': 'Federation',
   'start': 247,
   'end': 257}],
 [{'entity_group': 'MISC',
   'score': np.float32(0.9997856),
   'word': 'Russian',
   'start': 66,
   'end': 73},
  {'entity_group': 'LOC',
   'score': np

In [37]:
df_BERT = pd.DataFrame([
    {
        "page": example,
        "paragraph_id": paragraph_id,
        "paragraph": data_processed[example][paragraph_id],
        **entity
    }
    for paragraph_id, entities in enumerate(output_BERT)
    for entity in entities
])

df_BERT

,page,paragraph_id,paragraph,entity_group,score,word,start,end
0,Zapad 2021,0,"Zapad 2021 (Russian: Запад-2021, English: West...",MISC,0.390697,Zapad,0,5
1,Zapad 2021,0,"Zapad 2021 (Russian: Запад-2021, English: West...",MISC,0.996763,Russian,12,19
2,Zapad 2021,0,"Zapad 2021 (Russian: Запад-2021, English: West...",LOC,0.457594,West,42,46
3,Zapad 2021,0,"Zapad 2021 (Russian: Запад-2021, English: West...",LOC,0.999498,Russian Federation,116,134
4,Zapad 2021,0,"Zapad 2021 (Russian: Запад-2021, English: West...",LOC,0.999797,Belarus,139,146
...,...,...,...,...,...,...,...,...
127,Zapad 2021,19,According to American General and Chairman of ...,LOC,0.999749,Europe,627,633
128,Zapad 2021,20,Warsaw-based Belarusian former politician and ...,MISC,0.999306,Belarusian,13,23
129,Zapad 2021,20,Warsaw-based Belarusian former politician and ...,PER,0.963414,Andrej Sannikau,56,71
130,Zapad 2021,20,Warsaw-based Belarusian former politician and ...,MISC,0.564288,Zapad,89,94


### Model testing and evaluation (done on 10% of the data)

#### subset creation

create a 10% subset to evalute the models on the data.

In [38]:
all_pages = list(data_processed.keys())

if not all_pages:
    raise ValueError("data_processed contains no pages.")

sample_size = max(1, math.ceil(len(all_pages) * 0.10))

evaluation_pages = (
    pd.Series(all_pages)
    .sample(n=sample_size, random_state=42)
    .tolist()
)

evaluation_data = {
    page: data_processed[page]
    for page in evaluation_pages
}

print(f"Sampled {sample_size} of {len(all_pages)} pages.")

Sampled 90 of 897 pages.


#### DistilBERT

In [39]:

BATCH_SIZE = 32 if ner_DistilBERT.device.type == "cuda" else 1

# Keep each paragraph paired with its original page and paragraph ID.
paragraph_records = [
    (page, paragraph_id, paragraph)
    for page, paragraphs in evaluation_data.items()
    for paragraph_id, paragraph in enumerate(paragraphs)
]

rows = []
started = time.perf_counter()

# Stream paragraphs through the model in batches.
predictions = ner_DistilBERT(
    (record[2] for record in paragraph_records),
    batch_size=BATCH_SIZE,
    num_workers=0
)

for (page, paragraph_id, paragraph), entities in tqdm(
    zip(paragraph_records, predictions, strict=True),
    total=len(paragraph_records),
    desc="DistilBERT NER",
    unit="paragraph"
):
    for entity in entities:
        rows.append({
            "entity_group": entity["entity_group"],
            "score": entity["score"],
            "word": entity["word"],
            "start": entity["start"],
            "end": entity["end"],
            "Source": page,
            "Index": paragraph_id,
            "Text": paragraph
        })

elapsed = time.perf_counter() - started

columns = [
    "entity_group",
    "score",
    "word",
    "start",
    "end",
    "Source",
    "Index",
    "Text"
]

df_DistilBERT_10pct = pd.DataFrame(rows, columns=columns)

output_dir = os.path.join(
    "NER_output",
    "model_testing_and_evaluation"
)
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(
    output_dir,
    "post_NER_DistilBERT_10pct.csv"
)

df_DistilBERT_10pct.to_csv(output_path, index=False)

print(
    f"Processed {len(evaluation_data)} sampled pages "
    f"and {len(paragraph_records)} paragraphs in {elapsed:.1f} seconds."
)
print(f"Batch size: {BATCH_SIZE}")
print(f"Saved {len(df_DistilBERT_10pct)} entity rows to: {output_path}")

df_DistilBERT_10pct.head()

DistilBERT NER: 100%|██████████| 4746/4746 [00:43<00:00, 108.76paragraph/s]


Processed 90 sampled pages and 4746 paragraphs in 43.6 seconds.
Batch size: 32
Saved 35222 entity rows to: NER_output\model_testing_and_evaluation\post_NER_DistilBERT_10pct.csv


,entity_group,score,word,start,end,Source,Index,Text
0,MISC,0.687287,Investment,57,67,Executive Order 14071,0,"Executive Order 14071, officially titled Prohi..."
1,MISC,0.496020,Services,83,91,Executive Order 14071,0,"Executive Order 14071, officially titled Prohi..."
2,LOC,0.797773,Russian Federation,99,117,Executive Order 14071,0,"Executive Order 14071, officially titled Prohi..."
3,MISC,0.647296,Russian,143,150,Executive Order 14071,0,"Executive Order 14071, officially titled Prohi..."
4,ORG,0.770204,Federation,151,161,Executive Order 14071,0,"Executive Order 14071, officially titled Prohi..."


#### BERT

In [40]:
BATCH_SIZE = 32 if ner_BERT.device.type == "cuda" else 1

# Keep each paragraph paired with its original page and paragraph ID.
paragraph_records = [
    (page, paragraph_id, paragraph)
    for page, paragraphs in evaluation_data.items()
    for paragraph_id, paragraph in enumerate(paragraphs)
]

rows = []
started = time.perf_counter()

# Stream paragraphs through the model in batches.
predictions = ner_BERT(
    (record[2] for record in paragraph_records),
    batch_size=BATCH_SIZE,
    num_workers=0
)

for (page, paragraph_id, paragraph), entities in tqdm(
    zip(paragraph_records, predictions, strict=True),
    total=len(paragraph_records),
    desc="BERT NER",
    unit="paragraph"
):
    for entity in entities:
        rows.append({
            "entity_group": entity["entity_group"],
            "score": entity["score"],
            "word": entity["word"],
            "start": entity["start"],
            "end": entity["end"],
            "Source": page,
            "Index": paragraph_id,
            "Text": paragraph
        })

elapsed = time.perf_counter() - started

columns = [
    "entity_group",
    "score",
    "word",
    "start",
    "end",
    "Source",
    "Index",
    "Text"
]

df_BERT_10pct = pd.DataFrame(rows, columns=columns)

output_dir = os.path.join(
    "NER_output",
    "model_testing_and_evaluation"
)
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(
    output_dir,
    "post_NER_BERT_10pct.csv"
)

df_BERT_10pct.to_csv(output_path, index=False)

print(
    f"Processed {len(evaluation_data)} sampled pages "
    f"and {len(paragraph_records)} paragraphs in {elapsed:.1f} seconds."
)
print(f"Batch size: {BATCH_SIZE}")
print(f"Saved {len(df_BERT_10pct)} entity rows to: {output_path}")

df_BERT_10pct.head()

BERT NER: 100%|██████████| 4746/4746 [00:49<00:00, 96.68paragraph/s] 


Processed 90 sampled pages and 4746 paragraphs in 49.1 seconds.
Batch size: 32
Saved 32663 entity rows to: NER_output\model_testing_and_evaluation\post_NER_BERT_10pct.csv


,entity_group,score,word,start,end,Source,Index,Text
0,LOC,0.999146,Russian Federation,99,117,Executive Order 14071,0,"Executive Order 14071, officially titled Prohi..."
1,MISC,0.788013,Russian,143,150,Executive Order 14071,0,"Executive Order 14071, officially titled Prohi..."
2,LOC,0.608827,Federation,151,161,Executive Order 14071,0,"Executive Order 14071, officially titled Prohi..."
3,LOC,0.909937,U. S.,245,249,Executive Order 14071,0,"Executive Order 14071, officially titled Prohi..."
4,PER,0.964426,Joe Biden,260,269,Executive Order 14071,0,"Executive Order 14071, officially titled Prohi..."


### Full Data

Apply the chosen NER to the data, done post evaluation and improvement changes (improvement stage pending 11.10.26)

(currently DistilBert pending models evaluations)
date: 05.10.26

In [41]:
BATCH_SIZE = 32 if ner_DistilBERT.device.type == "cuda" else 1

# Use the full dataset.
paragraph_records = [
    (page, paragraph_id, paragraph)
    for page, paragraphs in data_processed.items()
    for paragraph_id, paragraph in enumerate(paragraphs)
]

rows = []
started = time.perf_counter()

predictions = ner_DistilBERT(
    (record[2] for record in paragraph_records),
    batch_size=BATCH_SIZE,
    num_workers=0
)

for (page, paragraph_id, paragraph), entities in tqdm(
    zip(paragraph_records, predictions, strict=True),
    total=len(paragraph_records),
    desc="DistilBERT NER — full data",
    unit="paragraph"
):
    for entity in entities:
        rows.append({
            "entity_group": entity["entity_group"],
            "score": entity["score"],
            "word": entity["word"],
            "start": entity["start"],
            "end": entity["end"],
            "Source": page,
            "Index": paragraph_id,
            "Text": paragraph
        })

elapsed = time.perf_counter() - started

columns = [
    "entity_group",
    "score",
    "word",
    "start",
    "end",
    "Source",
    "Index",
    "Text"
]

df_DistilBERT = pd.DataFrame(rows, columns=columns)

output_dir = "NER_output"
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, "post_NER.csv")
df_DistilBERT.to_csv(output_path, index=False)

print(
    f"Processed {len(data_processed)} pages "
    f"and {len(paragraph_records)} paragraphs in {elapsed:.1f} seconds."
)
print(f"Model device: {ner_DistilBERT.device}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Saved {len(df_DistilBERT)} entity rows to {output_path}")

DistilBERT NER — full data: 100%|██████████| 42822/42822 [06:13<00:00, 114.74paragraph/s]


Processed 897 pages and 42822 paragraphs in 373.2 seconds.
Model device: cuda:0
Batch size: 32
Saved 301525 entity rows to NER_output\post_NER.csv


In [45]:
MAX_BYTES = 90 * 1024 * 1024

source = Path(REPO_PATH) / "NER_output" / "post_NER.csv"
parts_dir = source.parent / "post_NER_parts"
parts_dir.mkdir(parents=True, exist_ok=True)

# Allow long text fields.
csv.field_size_limit(2**31 - 1)

def encode_row(row):
    buffer = io.StringIO()
    csv.writer(buffer, lineterminator="\n").writerow(row)
    return buffer.getvalue().encode("utf-8")

with source.open("r", encoding="utf-8", newline="") as f:
    reader = csv.reader(f)
    header = encode_row(next(reader))

    # Remove parts from an earlier run.
    for old_part in parts_dir.glob("part_*.csv"):
        old_part.unlink()

    part_number = 1
    output = (parts_dir / f"part_{part_number:04d}.csv").open("wb")
    output.write(header)
    current_size = len(header)
    row_count = 0

    try:
        for row in reader:
            encoded = encode_row(row)

            if len(header) + len(encoded) > MAX_BYTES:
                raise ValueError("A single CSV row exceeds the part size limit.")

            if current_size + len(encoded) > MAX_BYTES:
                output.close()
                part_number += 1
                output = (
                    parts_dir / f"part_{part_number:04d}.csv"
                ).open("wb")
                output.write(header)
                current_size = len(header)

            output.write(encoded)
            current_size += len(encoded)
            row_count += 1
    finally:
        output.close()

print(f"Saved {row_count} entity rows across {part_number} CSV parts.")
for part in sorted(parts_dir.glob("part_*.csv")):
    print(f"{part.name}: {part.stat().st_size / 1024**2:.2f} MiB")

Saved 301525 entity rows across 2 CSV parts.
part_0001.csv: 90.00 MiB
part_0002.csv: 87.99 MiB


## Commit to repo

In [46]:
GITHUB_USER = "idoschw3"
GITHUB_REPO = "Corpus2GeoRAG"
GITHUB_EMAIL = "193781032+idoschw3@users.noreply.github.com"
COMMIT_MESSAGE = "Update NER workflow"

LOCAL_REPO_PATH = Path.home() / GITHUB_REPO
MAX_GIT_FILE_BYTES = 100 * 1024 * 1024

# --------------------------------------------------
# Detect runtime
# --------------------------------------------------

if os.name == "nt":
    IS_LOCAL = True
elif os.environ.get("COLAB_RELEASE_TAG"):
    IS_LOCAL = False
else:
    raise RuntimeError(
        "Could not identify this runtime as Windows or a hosted Colab VM."
    )

if shutil.which("git") is None:
    raise RuntimeError(
        "Git is not available to this Jupyter environment. "
        "Install Git for Windows and restart Jupyter if running locally."
    )

# --------------------------------------------------
# Locate the repository
# --------------------------------------------------

if IS_LOCAL:
    candidates = [Path.cwd(), LOCAL_REPO_PATH]

    previous_repo_path = globals().get("REPO_PATH")
    if previous_repo_path:
        candidates.insert(0, Path(previous_repo_path))

    detected_repo = None

    for candidate in candidates:
        if not candidate.is_dir():
            continue

        probe = subprocess.run(
            ["git", "rev-parse", "--show-toplevel"],
            cwd=candidate,
            text=True,
            encoding="utf-8",
            errors="replace",
            capture_output=True
        )

        if probe.returncode == 0:
            root = Path(probe.stdout.strip()).resolve()

            if root.name.lower() == GITHUB_REPO.lower():
                detected_repo = root
                break

    if detected_repo is None:
        raise FileNotFoundError(
            f"Could not find your local {GITHUB_REPO} checkout.\n"
            f"Clone it to {LOCAL_REPO_PATH}, or update LOCAL_REPO_PATH."
        )

    REPO_PATH = str(detected_repo)

else:
    REPO_PATH = f"/content/{GITHUB_REPO}"

    if not Path(REPO_PATH).is_dir():
        raise FileNotFoundError(
            f"{REPO_PATH} does not exist. Run your clone cell first."
        )

os.chdir(REPO_PATH)

print("Runtime:", "Local Windows" if IS_LOCAL else "Colab VM")
print("Repository:", REPO_PATH)


def run_git(args, *, env=None, show_output=True):
    """Run Git and include its actual output if it fails."""
    result = subprocess.run(
        ["git", *args],
        cwd=REPO_PATH,
        env=env,
        stdin=subprocess.DEVNULL,
        text=True,
        encoding="utf-8",
        errors="replace",
        capture_output=True
    )

    if show_output:
        if result.stdout:
            print(result.stdout, end="")
        if result.stderr:
            print(result.stderr, end="")

    if result.returncode != 0:
        details = (result.stderr or result.stdout or "").strip()
        raise RuntimeError(f"git {' '.join(args)} failed:\n{details}")

    return result


# --------------------------------------------------
# Git configuration
# --------------------------------------------------

run_git(["config", "user.name", GITHUB_USER])
run_git(["config", "user.email", GITHUB_EMAIL])
run_git([
    "remote", "set-url", "origin",
    f"https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
])

branch = run_git(
    ["branch", "--show-current"],
    show_output=False
).stdout.strip()

if not branch:
    raise RuntimeError("Could not determine the current Git branch.")

print("Branch:", branch)

# Stop if an earlier rebase or merge is still paused.
for state_path in ("rebase-merge", "rebase-apply", "MERGE_HEAD"):
    git_state = run_git(
        ["rev-parse", "--git-path", state_path],
        show_output=False
    ).stdout.strip()

    if Path(git_state).exists():
        raise RuntimeError(
            "An earlier rebase or merge is still in progress. "
            "Check `git status` and resolve it before rerunning this cell."
        )

# --------------------------------------------------
# Authentication and synchronization
# --------------------------------------------------

git_env = os.environ.copy()
git_env["GIT_TERMINAL_PROMPT"] = "0"

auth_options = []
askpass_dir = None

try:
    if IS_LOCAL:
        # Use Git credentials already configured on your computer.
        print("Authentication: local Git credentials.")

    else:
        from google.colab import userdata

        github_token = userdata.get("GITHUB_TOKEN")

        if not github_token:
            raise ValueError(
                "GITHUB_TOKEN is missing from Colab Secrets. "
                "Add it and enable notebook access."
            )

        askpass_dir = tempfile.TemporaryDirectory(
            prefix="github-askpass-"
        )
        askpass_path = Path(askpass_dir.name) / "askpass.sh"

        askpass_path.write_text(
            "#!/bin/sh\n"
            'case "$1" in\n'
            '  *Username*) printf "%s\\n" "$GITHUB_USER" ;;\n'
            '  *Password*) printf "%s\\n" "$GITHUB_TOKEN" ;;\n'
            "esac\n",
            encoding="utf-8"
        )
        askpass_path.chmod(0o700)

        git_env["GITHUB_USER"] = GITHUB_USER
        git_env["GITHUB_TOKEN"] = github_token
        git_env["GIT_ASKPASS"] = str(askpass_path)

        auth_options = ["-c", "credential.helper="]

        print("Authentication: Colab GITHUB_TOKEN Secret.")

    # --------------------------------------------------
    # Prevent accidental nested repository
    # --------------------------------------------------

    ignore_entry = f"/{GITHUB_REPO}/"
    gitignore_path = Path(REPO_PATH) / ".gitignore"

    existing_ignore = (
        gitignore_path.read_text(encoding="utf-8")
        if gitignore_path.exists()
        else ""
    )

    if ignore_entry not in existing_ignore.splitlines():
        with gitignore_path.open("a", encoding="utf-8") as f:
            if existing_ignore and not existing_ignore.endswith("\n"):
                f.write("\n")
            f.write(ignore_entry + "\n")

    nested_repo = run_git(
        ["ls-files", "--stage", "--", GITHUB_REPO],
        show_output=False
    ).stdout.strip()

    if nested_repo:
        run_git([
            "rm", "--cached", "-r", "-f", "--", GITHUB_REPO
        ])
        print("Removed the nested repository path from Git tracking.")

    # --------------------------------------------------
    # Find files exceeding GitHub's per-file limit
    # --------------------------------------------------

    candidate_files = run_git(
        [
            "ls-files",
            "--cached",
            "--others",
            "--exclude-standard",
            "-z"
        ],
        show_output=False
    ).stdout.split("\0")

    excluded_files = set()

    for relative_path in sorted(set(candidate_files) - {""}):
        file_path = Path(REPO_PATH) / relative_path

        if not file_path.is_file():
            continue

        file_size = file_path.stat().st_size

        if file_size > MAX_GIT_FILE_BYTES:
            excluded_files.add(relative_path)

            # Remove from Git's index, while keeping the file on disk.
            run_git(
                [
                    "rm",
                    "--cached",
                    "--ignore-unmatch",
                    "-f",
                    "--",
                    f":(top,literal){relative_path}"
                ],
                show_output=False
            )

            print(
                f"Excluded oversized file: {relative_path} "
                f"({file_size / 1024**2:.2f} MiB). "
                "Kept on disk."
            )

    # --------------------------------------------------
    # Stage changes, excluding oversized files
    # --------------------------------------------------

    exclusion_pathspecs = [
        f":(top,literal,exclude){path}"
        for path in sorted(excluded_files)
    ]

    run_git(["add", "-A", "--", ".", *exclusion_pathspecs])

    # Exclude notebook copies, preserving the original cell's behavior.
    staged_files = run_git(
        ["diff", "--cached", "--name-only", "-z"],
        show_output=False
    ).stdout.split("\0")

    for notebook in staged_files:
        if (
            notebook.endswith(".ipynb")
            and notebook not in excluded_files
        ):
            run_git([
                "restore", "--staged", "--",
                f":(top,literal){notebook}"
            ])

    # --------------------------------------------------
    # Create a normal commit
    # --------------------------------------------------

    staged_diff = subprocess.run(
        ["git", "diff", "--cached", "--quiet"],
        cwd=REPO_PATH,
        stdin=subprocess.DEVNULL
    )

    if staged_diff.returncode == 1:
        run_git(["commit", "-m", COMMIT_MESSAGE])
        print("Repository changes committed.")
    elif staged_diff.returncode == 0:
        print("No new repository changes to commit.")
    else:
        raise RuntimeError("Could not check the staged changes.")

    # --------------------------------------------------
    # Restore tracked notebook copies on disk
    # This discards edits to those on-disk copies.
    # --------------------------------------------------

    tracked_files = run_git(
        ["ls-tree", "-r", "--name-only", "-z", "HEAD"],
        show_output=False
    ).stdout.split("\0")

    for notebook in tracked_files:
        if notebook.endswith(".ipynb"):
            run_git([
                "restore", "--worktree", "--",
                f":(top,literal){notebook}"
            ])

    # --------------------------------------------------
    # Pull/rebase GitHub changes
    # --------------------------------------------------

    pull = subprocess.run(
        ["git", *auth_options, "pull", "--rebase", "origin", branch],
        cwd=REPO_PATH,
        env=git_env,
        stdin=subprocess.DEVNULL,
        text=True,
        encoding="utf-8",
        errors="replace",
        capture_output=True
    )

    if pull.stdout:
        print(pull.stdout, end="")
    if pull.stderr:
        print(pull.stderr, end="")

    if pull.returncode != 0:
        conflicted = run_git(
            ["diff", "--name-only", "--diff-filter=U"],
            show_output=False
        ).stdout.splitlines()

        if conflicted == [".gitignore"]:
            ours = run_git(
                ["show", ":2:.gitignore"],
                show_output=False
            ).stdout

            theirs = run_git(
                ["show", ":3:.gitignore"],
                show_output=False
            ).stdout

            if ours.strip() != theirs.strip():
                raise RuntimeError(
                    "The .gitignore versions differ. The rebase is paused; "
                    "inspect the conflict before continuing."
                )

            gitignore_path.write_text(
                ours + ("\n" if ours and not ours.endswith("\n") else ""),
                encoding="utf-8"
            )

            run_git(["add", "--", ".gitignore"])
            run_git([
                "-c", "core.editor=true", "rebase", "--continue"
            ])

        else:
            raise RuntimeError(
                "Pull failed. Check the Git output above and `git status`. "
                "If running locally, verify your GitHub authentication."
            )

    # --------------------------------------------------
    # Push synchronized changes
    # --------------------------------------------------

    run_git(
        [*auth_options, "push", "origin", branch],
        env=git_env
    )

    print(
        f"Successfully pushed to {GITHUB_USER}/{GITHUB_REPO} "
        f"on branch '{branch}'."
    )

finally:
    if askpass_dir is not None:
        askpass_dir.cleanup()

    git_env.pop("GITHUB_TOKEN", None)

Runtime: Local Windows
Repository: C:\Users\idosc\Corpus2GeoRAG
Branch: main
Authentication: local Git credentials.
Excluded oversized file: NER_output/post_NER.csv (178.28 MiB). Kept on disk.
[main d0ccc0d] Update NER workflow
 2 files changed, 301527 insertions(+)
 create mode 100644 NER_output/post_NER_parts/part_0001.csv
 create mode 100644 NER_output/post_NER_parts/part_0002.csv
Repository changes committed.
Current branch main is up to date.
From https://github.com/idoschw3/Corpus2GeoRAG
 * branch            main       -> FETCH_HEAD
remote: warning: See https://gh.io/lfs for more information.        
remote: warning: File NER_output/post_NER_parts/part_0002.csv is 87.99 MB; this is larger than GitHub's recommended maximum file size of 50.00 MB        
remote: warning: File NER_output/post_NER_parts/part_0001.csv is 90.00 MB; this is larger than GitHub's recommended maximum file size of 50.00 MB        
remote: warning: GH001: Large files detected. You may want to try Git Large Fi